# 🐍 Laya System-1 Decision Agent: 貪吃蛇 (Snake Game) Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook surveys and demonstrates **[Child-pi/laya](https://github.com/Child-pi/laya)** (Node.js/TypeScript port of **Convai Innovations' Laya System-1 Decision Model** by Satoshi Nakajima) and applies it to govern the real-time decisions of a **貪吃蛇 (Snake Game)** agent.

---

## 🌟 Core Concepts: What is Laya?

1. **Non-Autoregressive System-1 AI**:
   Unlike traditional generative LLMs (GPT, Claude, LLaMA) that generate text token-by-token (taking seconds), Laya does not generate text. You hand it an arbitrary **structured state** (JSON / text) and **typed questions**, and it produces all decisions with mathematically calibrated probabilities in a **single forward pass** (~30–50 ms).

2. **Three Decision Primitives**:
   - `choice`: Pick one discrete option (e.g. `UP`, `DOWN`, `LEFT`, `RIGHT`) with normalized softmax probabilities.
   - `score`: Expected value on an ordered rubric scale (e.g. `danger_level` from `0: safe` to `3: deadly`).
   - `noul`: Calibrated binary probability $P(\text{true}) \in [0, 1]$ (e.g. `is_food_reachable`).

3. **Architecture & Runtime**:
   - **Base Encoder**: ModernBERT encoder trained with Reinforcement Learning (Jev-compatible).
   - **Exported Weights**: Hosted at Hugging Face [`receptron/laya-onnx`](https://huggingface.co/receptron/laya-onnx) (~1.7 GB ONNX bundle).
   - **Child-pi/laya**: TypeScript/Node.js implementation using `onnxruntime-node` and `@huggingface/tokenizers` (no PyTorch required).

---

## 🎮 How the Snake Agent Works
At each step of the snake simulation:
1. **Perception**: The grid is scanned. The snake's head coordinates, food coordinates, and a 1-step lookahead for obstacles (walls, self-body) are compiled into a JSON state.
2. **System-1 Inference**: Laya receives the state and 3 questions simultaneously:
   - `next_move` (`choice`): Which direction (`UP`, `DOWN`, `LEFT`, `RIGHT`) to move?
   - `danger_level` (`score`): How dangerous are current surroundings?
   - `viable_path` (`noul`): Is there a safe path open?
3. **Execution**: The snake moves according to Laya's highest-probability action.

We provide two running options below:
- **Option 1**: Directly run `Child-pi/laya` via Node.js / TypeScript in Colab.
- **Option 2**: Run via Python with an interactive animated visual HTML5 Canvas dashboard in Colab.

--- 
## 🚀 Option 1: Run Child-pi/laya in TypeScript (Node.js)

Google Colab includes Node.js pre-installed. We clone the repository, install dependencies, and run the TypeScript game demo.

In [ ]:
# Clone Child-pi/laya and install packages
!git clone https://github.com/Child-pi/laya.git
%cd laya
!npm install
!npm install -g tsx

In [ ]:
# Run the TypeScript Snake demo
# On first run, it downloads the ONNX bundle from receptron/laya-onnx on Hugging Face (~10-15 seconds in Colab)
!npx tsx examples/snake_demo.ts

---
## 🎨 Option 2: Pure Python Interactive Visual Snake Demo

If you prefer a native Python experience inside Colab with an **animated visual display**, this section loads the exact same Laya ONNX weights via `onnxruntime` and renders a live animated game with real-time probability distributions!

In [ ]:
# Install Python dependencies
!pip install -q onnxruntime huggingface_hub transformers numpy

In [ ]:
import os
import json
import math
import time
import numpy as np
import onnxruntime as ort
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from IPython.display import clear_output, display, HTML

# 1. Download ONNX bundle from Hugging Face
print("📥 Downloading / verifying Laya ONNX bundle from Hugging Face (receptron/laya-onnx)...")
model_dir = snapshot_download(
    repo_id="receptron/laya-onnx",
    allow_patterns=["laya.onnx", "laya.onnx.data", "laya_config.json", "tokenizer/*"]
)
print(f"✅ Model bundle ready at: {model_dir}")

In [ ]:
class PythonLayaAgent:
    """
    Python implementation of Laya's System-1 decision pipeline,
    fully compatible with Child-pi/laya and convaiinnovations/laya.
    """
    QTYPES = {"choice": 0, "score": 1, "noul": 2}
    QTYPE_NAMES = ["choice", "score", "noul"]

    def __init__(self, model_dir):
        with open(os.path.join(model_dir, "laya_config.json")) as f:
            self.config = json.load(f)
        
        self.tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
        self.cls_id = self.tok.cls_token_id or self.tok.convert_tokens_to_ids("[CLS]")
        self.sep_id = self.tok.sep_token_id or self.tok.convert_tokens_to_ids("[SEP]")
        self.mask_id = self.tok.mask_token_id or self.tok.convert_tokens_to_ids("[MASK]")
        self.pad_id = self.tok.pad_token_id or self.tok.convert_tokens_to_ids("[PAD]")
        
        # Initialize ONNX Runtime session
        so = ort.SessionOptions()
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session = ort.InferenceSession(
            os.path.join(model_dir, "laya.onnx"),
            sess_options=so,
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
        )
        print(f"✅ Laya ONNX Session loaded using provider: {self.session.get_providers()[0]}")

    def _render_options(self, qtype, criteria):
        if qtype == "choice":
            return [f"{k}: {v}" if v else k for k, v in criteria.items()]
        elif qtype == "score":
            return [f"level {i}: {c}" for i, c in enumerate(criteria)]
        else:
            return ["false: no, the statement does not hold", "true: yes, the statement holds"]

    def _build_sequence(self, state, qtype, instructions, criteria):
        max_len = self.config["max_len"]
        head_max_len = self.config["head_max_len"]
        
        opts = self._render_options(qtype, criteria)
        head_text = f"{qtype} question: {instructions}"
        head_ids = self.tok.encode(head_text, add_special_tokens=False)
        
        opt_ids = [[self.mask_id] + self.tok.encode(" " + o, add_special_tokens=False)[:48] for o in opts]
        total_opts = sum(len(o) for o in opt_ids)
        opt_budget = head_max_len - total_opts
        if opt_budget < 16:
            per = max(4, (head_max_len - 16) // max(1, len(opt_ids)))
            opt_ids = [o[:per] for o in opt_ids]
            opt_budget = head_max_len - sum(len(o) for o in opt_ids)
        
        head_ids = head_ids[:max(8, opt_budget)]
        seq = [self.cls_id] + head_ids + [self.sep_id]
        markers = []
        for o in opt_ids:
            markers.append(len(seq))
            seq.extend(o)
        seq.append(self.sep_id)
        
        room = max(0, max_len - len(seq) - 1)
        state_str = json.dumps(state, ensure_ascii=False) if not isinstance(state, str) else state
        st_ids = self.tok.encode(state_str, add_special_tokens=False)[:room]
        seq.extend(st_ids)
        seq.append(self.sep_id)
        
        return seq[:max_len], [m for m in markers if m < max_len]

    def system_one(self, state, questions):
        qids = list(questions.keys())
        items = []
        for qid in qids:
            q = questions[qid]
            qtype = q["type"]
            ins = q["instructions"]
            crit = q.get("criteria")
            ids, markers = self._build_sequence(state, qtype, ins, crit)
            items.append({
                "qid": qid,
                "qtype": qtype,
                "qtype_num": self.QTYPES[qtype],
                "crit": crit,
                "ids": ids,
                "markers": markers
            })
        
        n = len(items)
        L = max(len(it["ids"]) for it in items)
        K = max(len(it["markers"]) for it in items)
        
        input_ids = np.full((n, L), self.pad_id, dtype=np.int64)
        attention_mask = np.zeros((n, L), dtype=np.int64)
        marker_pos = np.zeros((n, K), dtype=np.int64)
        marker_mask = np.zeros((n, K), dtype=bool)
        qtype_arr = np.zeros((n,), dtype=np.int64)
        
        for i, it in enumerate(items):
            seq_len = len(it["ids"])
            input_ids[i, :seq_len] = it["ids"]
            attention_mask[i, :seq_len] = 1
            m_len = len(it["markers"])
            marker_pos[i, :m_len] = it["markers"]
            marker_mask[i, :m_len] = True
            qtype_arr[i] = it["qtype_num"]
            
        # Execute ONNX graph in single pass
        logits, act_probs = self.session.run(None, {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "marker_pos": marker_pos,
            "marker_mask": marker_mask,
            "qtype": qtype_arr,
        })
        
        answers = {}
        for i, it in enumerate(items):
            qid = it["qid"]
            k = len(it["markers"])
            qtype = it["qtype"]
            
            # Temperature scaling
            sz_b = "2" if k <= 2 else "3-5" if k <= 5 else "6-10" if k <= 10 else "11+"
            b_key = f"{qtype}:{sz_b}"
            temp = self.config["temperature_by_options"].get(b_key, self.config["temperature"][it["qtype_num"]])
            
            raw_logits = logits[i, :k] / temp
            e = np.exp(raw_logits - np.max(raw_logits))
            p = (e / np.sum(e)).tolist()
            
            # Confidence (1 - normalized entropy)
            if k < 2:
                conf = 1.0
            else:
                ent = -sum(x * math.log(max(x, 1e-12)) for x in p)
                conf = 1.0 - ent / math.log(k)
                
            if qtype == "choice":
                keys = list(it["crit"].keys())
                best_idx = int(np.argmax(p))
                answers[qid] = {
                    "type": "choice",
                    "choice": keys[best_idx],
                    "probabilities": {keys[j]: round(p[j], 4) for j in range(len(keys))},
                    "confidence": round(conf, 4)
                }
            elif qtype == "score":
                ev = sum(j * p[j] for j in range(len(p)))
                answers[qid] = {
                    "type": "score",
                    "score": round(ev, 4),
                    "probabilities": {str(j): round(p[j], 4) for j in range(len(p))},
                    "confidence": round(conf, 4)
                }
            else:
                answers[qid] = {
                    "type": "noul",
                    "noul": round(p[1], 4)
                }
        return {"answers": answers, "latency_ms": 0}

In [ ]:
# 2. Define Snake Game Engine with Lookahead Perception
import random

class SnakeSimulator:
    DELTAS = {
        "UP": (0, -1),
        "DOWN": (0, 1),
        "LEFT": (-1, 0),
        "RIGHT": (1, 0)
    }
    OPPOSITES = {"UP": "DOWN", "DOWN": "UP", "LEFT": "RIGHT", "RIGHT": "LEFT"}

    def __init__(self, width=8, height=8):
        self.width = width
        self.height = height
        mid_x, mid_y = width // 2, height // 2
        self.snake = [(mid_x, mid_y), (mid_x - 1, mid_y), (mid_x - 2, mid_y)]
        self.score = 0
        self.steps = 0
        self.alive = True
        self.last_direction = "RIGHT"
        self.food = self._spawn_food()

    def _spawn_food(self):
        empty = [(x, y) for y in range(self.height) for x in range(self.width) if (x, y) not in self.snake]
        return random.choice(empty) if empty else (-1, -1)

    def is_collision(self, pt):
        x, y = pt
        if x < 0 or x >= self.width or y < 0 or y >= self.height:
            return True
        return pt in self.snake[:-1]

    def get_state(self):
        hx, hy = self.snake[0]
        fx, fy = self.food
        analysis = {}
        for d, (dx, dy) in self.DELTAS.items():
            nxt = (hx + dx, hy + dy)
            collides = self.is_collision(nxt)
            is_reverse = (d == self.OPPOSITES[self.last_direction]) and len(self.snake) > 1
            dist = abs(nxt[0] - fx) + abs(nxt[1] - fy)
            if collides or is_reverse:
                analysis[d] = f"DEADLY: {'reverse into body' if is_reverse else 'collision'}"
            else:
                analysis[d] = f"safe path, distance to food = {dist}"
        
        return {
            "game": "Snake",
            "grid": f"{self.width}x{self.height}",
            "head": [hx, hy],
            "food": [fx, fy],
            "steps": self.steps,
            "score": self.score,
            "options_analysis": analysis
        }

    def step(self, direction):
        if not self.alive:
            return False, False
        dx, dy = self.DELTAS[direction]
        hx, hy = self.snake[0]
        new_head = (hx + dx, hy + dy)
        
        if self.is_collision(new_head):
            self.alive = False
            return False, False
            
        self.snake.insert(0, new_head)
        self.last_direction = direction
        self.steps += 1
        
        ate_food = (new_head == self.food)
        if ate_food:
            self.score += 1
            self.food = self._spawn_food()
        else:
            self.snake.pop()
        return True, ate_food

In [ ]:
# 3. Run Live Interactive Simulation in Colab with HTML Dashboard

agent = PythonLayaAgent(model_dir)
game = SnakeSimulator(width=8, height=8)

def render_html_dashboard(game, result, latency_ms):
    # Grid rendering
    cells_html = ""
    for y in range(game.height):
        cells_html += "<div style='display: flex;'>"
        for x in range(game.width):
            pt = (x, y)
            if pt == game.snake[0]:
                color = "#10b981" if game.alive else "#ef4444"
                symbol = "👀" if game.alive else "💥"
            elif pt in game.snake:
                color = "#34d399"
                symbol = ""
            elif pt == game.food:
                color = "#f87171"
                symbol = "🍎"
            else:
                color = "#1e293b"
                symbol = ""
            cells_html += f"""<div style="
                width: 38px; height: 38px; margin: 2px;
                background: {color}; border-radius: 6px;
                display: flex; align-items: center; justify-content: center;
                font-size: 20px;">{symbol}</div>"""
        cells_html += "</div>"
        
    choice = result["answers"]["next_move"]["choice"]
    conf = result["answers"]["next_move"]["confidence"] * 100
    probs = result["answers"]["next_move"]["probabilities"]
    danger = result["answers"]["danger_level"]["score"]
    path_ok = result["answers"]["viable_path"]["noul"] * 100
    
    prob_bars = ""
    for d in ["UP", "DOWN", "LEFT", "RIGHT"]:
        p_val = probs.get(d, 0.0) * 100
        highlight = "#22c55e" if d == choice else "#64748b"
        prob_bars += f"""
        <div style="margin: 6px 0;">
          <div style="display: flex; justify-content: space-between; font-size: 13px; font-weight: bold;">
            <span>{d}</span><span>{p_val:.1f}%</span>
          </div>
          <div style="background: #334155; height: 8px; border-radius: 4px; overflow: hidden;">
            <div style="width: {p_val}%; height: 100%; background: {highlight};"></div>
          </div>
        </div>
        """

    html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0f172a; color: #f8fafc; padding: 20px; border-radius: 12px; max-width: 650px;">
      <h2 style="margin: 0 0 10px 0; color: #38bdf8;">🐍 Laya System-1 Snake Decision Controller</h2>
      <div style="display: flex; gap: 24px;">
        <div>{cells_html}</div>
        <div style="flex: 1;">
          <div style="background: #1e293b; padding: 12px; border-radius: 8px; margin-bottom: 12px;">
            <div style="font-size: 14px; color: #94a3b8;">Game Status</div>
            <div style="font-size: 18px; font-weight: bold;">Score: <span style="color: #fbbf24;">{game.score}</span> | Step: {game.steps}</div>
            <div style="font-size: 12px; color: #a5b4fc;">Single-pass Latency: <b>{latency_ms:.1f} ms</b></div>
          </div>
          <div style="background: #1e293b; padding: 12px; border-radius: 8px;">
            <div style="font-size: 14px; color: #94a3b8; margin-bottom: 4px;">Laya Decisions</div>
            <div style="font-size: 16px;">Move: <b style="color: #4ade80;">{choice}</b> (Conf: {conf:.1f}%)</div>
            <div style="font-size: 13px; color: #cbd5e1;">Danger Level: <b>{danger:.2f}/3.0</b> | Path Viability: <b>{path_ok:.1f}%</b></div>
            <hr style="border-color: #334155; margin: 10px 0;">
            <div style="font-size: 12px; color: #94a3b8; margin-bottom: 4px;">Action Probabilities:</div>
            {prob_bars}
          </div>
        </div>
      </div>
    </div>
    """
    clear_output(wait=True)
    display(HTML(html))

# Run game loop
max_steps = 30
while game.alive and game.steps < max_steps:
    state = game.get_state()
    
    t0 = time.perf_counter()
    result = agent.system_one(state, {
        "next_move": {
            "type": "choice",
            "instructions": "Choose the safest direction to avoid walls and navigate toward food.",
            "criteria": state["options_analysis"]
        },
        "danger_level": {
            "type": "score",
            "instructions": "Assess the immediate collision danger level.",
            "criteria": ["safe open space", "minor hazard", "high risk", "fatal trap"]
        },
        "viable_path": {
            "type": "noul",
            "instructions": "Is there an unobstructed path forward?"
        }
    })
    latency = (time.perf_counter() - t0) * 1000
    
    render_html_dashboard(game, result, latency)
    
    chosen_move = result["answers"]["next_move"]["choice"]
    alive, ate = game.step(chosen_move)
    
    if not alive:
        render_html_dashboard(game, result, latency)
        print("💥 Game Over: Snake collided!")
        break
    time.sleep(0.35)

print(f"\nGame concluded! Final Score: {game.score}, Steps: {game.steps}")

---
## 💡 Observations & Key Findings

1. **Inference Latency**:
   Notice that answering all 3 queries (`choice`, `score`, `noul`) takes **only ~30-40 ms** on CPU/GPU. This is because all questions and options are collapsed into a single sequence and executed in a single forward pass.
2. **Safety & Progress**:
   Because Laya's ModernBERT encoder attends to both the perceptual state (food distance, coordinate offsets) and the semantic options (`safe path` vs `DEADLY: collision`), Laya strongly suppresses lethal moves and chooses the path minimizing distance to the apple.
3. **Dual System Integration**:
   Laya provides an ideal **System-1** layer (reflexive, fast, high-throughput). When confidence drops below a threshold (e.g. `conf < 0.4`), a slower **System-2** planner (e.g. A* search or an autoregressive LLM) can be triggered.